In [ ]:

# ==========================================
# BERT CUSTOMER FEEDBACK CLASSIFIER
# ==========================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ==========================================
# 1. DEVICE CONFIGURATION
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==========================================
# 2. LOAD DATASET
# ==========================================

df = pd.read_csv("Customer_Sentiment.csv")

print("\n========== DATASET INFORMATION ==========")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 Rows:\n", df.head())

# ==========================================
# 3. SELECT TEXT AND LABEL COLUMNS
# ==========================================

df = df[["review_text", "sentiment"]].copy()

df.columns = ["text", "sentiment"]

print("\nText Column: review_text")
print("Label Column: sentiment")

# ==========================================
# 4. DATA CLEANING
# ==========================================

df = df.dropna()

df["text"] = df["text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()

df = df[(df["text"] != "") & (df["sentiment"] != "")]

# Map negative sentiment to Complaint
df["sentiment"] = df["sentiment"].replace({
    "negative": "Complaint",
    "neutral": "Neutral",
    "positive": "Positive"
})

print("\n========== CLEANED DATASET ==========")
print("Shape:", df.shape)
print(df.head())

# ==========================================
# 5. SENTIMENT DISTRIBUTION
# ==========================================

print("\n========== SENTIMENT DISTRIBUTION ==========")
print(df["sentiment"].value_counts())

# ==========================================
# 6. LABEL ENCODING
# ==========================================

label2id = {
    "Complaint": 0,
    "Neutral": 1,
    "Positive": 2
}

id2label = {
    0: "Complaint",
    1: "Neutral",
    2: "Positive"
}

df["labels"] = df["sentiment"].map(label2id)

print("\n========== LABEL ENCODING ==========")

for label, number in label2id.items():
    print(label, "->", number)

print("Number of Classes:", len(label2id))

# ==========================================
# 7. TRAIN TEST SPLIT
# ==========================================

# Smaller sample for CPU training.
# Change to None to use the entire dataset.

MAX_SAMPLES = 5000

if MAX_SAMPLES is not None and len(df) > MAX_SAMPLES:
    df = df.groupby("sentiment", group_keys=False).apply(
        lambda x: x.sample(
            n=max(1, round(MAX_SAMPLES * len(x) / len(df))),
            random_state=42
        )
    ).reset_index(drop=True)

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["labels"]
)

print("\n========== TRAIN TEST SPLIT ==========")
print("Training Samples:", len(train_df))
print("Testing Samples:", len(test_df))

# ==========================================
# 8. LOAD BERT TOKENIZER
# ==========================================

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("\nBERT Tokenizer Loaded")

# ==========================================
# 9. TOKENIZATION
# ==========================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=96
    )

train_dataset = Dataset.from_pandas(
    train_df[["text", "labels"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "labels"]].reset_index(drop=True)
)

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

train_dataset.set_format("torch")
test_dataset.set_format("torch")

print("\nTokenization Completed")
print("PyTorch Dataset Ready")

# ==========================================
# 10. LOAD BERT MODEL
# ==========================================

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("\n========== BERT MODEL LOADED ==========")

# ==========================================
# 11. EVALUATION METRICS
# ==========================================

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# ==========================================
# 12. TRAINING ARGUMENTS
# ==========================================

training_args = TrainingArguments(
    output_dir="./bert_customer_results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
    dataloader_pin_memory=False
)

# ==========================================
# 13. CREATE TRAINER
# ==========================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# ==========================================
# 14. TRAIN BERT MODEL
# ==========================================

print("\n========== STARTING BERT TRAINING ==========")

trainer.train()

print("\n========== TRAINING COMPLETED ==========")

# ==========================================
# 15. MODEL EVALUATION
# ==========================================

print("\n========== MODEL EVALUATION ==========")

results = trainer.evaluate()

for key, value in results.items():
    if isinstance(value, (int, float)):
        print(key, ":", round(value, 4))

# ==========================================
# 16. CLASSIFICATION REPORT
# ==========================================

predictions_output = trainer.predict(test_dataset)

y_pred = np.argmax(
    predictions_output.predictions,
    axis=1
)

y_true = predictions_output.label_ids

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=["Complaint", "Neutral", "Positive"],
        zero_division=0
    )
)

# ==========================================
# 17. CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2]
)

print("\n========== CONFUSION MATRIX ==========")
print(cm)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Complaint", "Neutral", "Positive"]
)

display.plot(cmap="Blues", values_format="d")

plt.title("BERT Customer Feedback Confusion Matrix")
plt.show()

# ==========================================
# 18. SAVE TRAINED BERT MODEL
# ==========================================

model_path = "./saved_bert_customer_model"

trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

with open(
    os.path.join(model_path, "label_mapping.json"),
    "w"
) as f:
    json.dump(label2id, f)

print("\n========== MODEL SAVED ==========")
print("Saved Location:", model_path)

# ==========================================
# 19. LOAD SAVED MODEL FOR PREDICTION
# ==========================================

saved_model = BertForSequenceClassification.from_pretrained(
    model_path
)

saved_tokenizer = BertTokenizer.from_pretrained(
    model_path
)

saved_model.to(device)
saved_model.eval()

# ==========================================
# 20. PREDICT CUSTOMER FEEDBACK
# ==========================================

def predict_feedback(text):

    inputs = saved_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=96
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = saved_model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

    predicted_id = torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence = probabilities[0][predicted_id].item() * 100

    predicted_class = id2label[predicted_id]

    print("\nInput:", text)
    print("Predicted Class:", predicted_class)
    print("Confidence:", round(confidence, 2), "%")

    return predicted_class, confidence

# ==========================================
# 21. TEST NEW CUSTOMER FEEDBACK
# ==========================================

print("\n========== CUSTOMER FEEDBACK PREDICTION ==========")

sample_text = "The product arrived damaged and I want a replacement."

predict_feedback(sample_text)

# ==========================================
# 22. CUSTOM USER INPUT
# ==========================================

while True:

    text = input("\nEnter customer feedback (or 'exit'): ")

    if text.lower() == "exit":
        print("Prediction stopped.")
        break

    predict_feedback(text)


Device: cpu

========== DATASET INFORMATION ==========
Shape: (25000, 13)
Columns: ['customer_id', 'gender', 'age_group', 'region', 'product_category', 'purchase_channel', 'platform', 'customer_rating', 'review_text', 'sentiment', 'response_time_hours', 'issue_resolved', 'complaint_registered']

First 5 Rows:
    customer_id  gender age_group   region product_category purchase_channel  \
0            1    male       60+    north       automobile           online   
1            2   other     46-60  central            books           online   
2            3  female     36-45     east           sports           online   
3            4  female     18-25  central        groceries           online   
4            5  female     18-25     east      electronics           online   

               platform  customer_rating  \
0              flipkart                1   
1      swiggy instamart                5   
2  facebook marketplace                1   
3                 zepto              


BERT Tokenizer Loaded


Map: 100%|██████████| 1000/1000 [00:00<00:00, 6733.29 examples/s]



Tokenization Completed
PyTorch Dataset Ready


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2256.70it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi


========== BERT MODEL LOADED ==========

========== STARTING BERT TRAINING ==========


Epoch,Training Loss,Validation Loss
